# Task 3: Lexical-Pair-Second Experiment

This notebook keeps the lexical-pair-second corpus construction, export, and evaluation flow, but replaces the staged training schedule with a plain Inside-Outside EM loop.

The intended shape is exactly the pattern you described:
- `A1 -> N1 N2`
- `N1 -> A B | C D`
- `N2 -> E F | G H`

Pipeline:
1. Convert the raw `a..z` corpus into Task 3 symbol sequences (`A..Z`, with `PT_S` for `s`).
2. Collapse the raw corpus into the curated lexical-pair inventory (`LP_AB`, `LP_QR`, `LP_ST`, ...).
3. Map those pair symbols into a smaller family inventory (`F_ABCD`, `F_QRST`, ...).
4. Train a simple EM PCFG on the family-symbol corpus.
5. Export a CNF grammar where every family symbol expands directly to one of its member base pairs.


In [ ]:
import collections
import csv
import time
from pathlib import Path

import numpy as np


BASE_PRETERMS = [chr(ord('A') + i) for i in range(26)]
BASE_PRETERMS[ord('s') - ord('a')] = 'PT_S'
WORD_TO_SYMBOL = {chr(ord('a') + i): BASE_PRETERMS[i] for i in range(26)}
SYMBOL_TO_WORD = {symbol: word for word, symbol in WORD_TO_SYMBOL.items()}


def load_symbol_corpus(path):
    corpus = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            toks = line.strip().split()
            if toks:
                corpus.append([WORD_TO_SYMBOL[tok] for tok in toks])
    return corpus


def count_adjacent_pairs(corpus):
    counts = collections.Counter()
    for sent in corpus:
        for i in range(len(sent) - 1):
            counts[(sent[i], sent[i + 1])] += 1
    return counts


def replace_pair_once(corpus, pair, new_symbol):
    updated = []
    replacements = 0
    left, right = pair
    for sent in corpus:
        out = []
        i = 0
        while i < len(sent):
            if i < len(sent) - 1 and sent[i] == left and sent[i + 1] == right:
                out.append(new_symbol)
                replacements += 1
                i += 2
            else:
                out.append(sent[i])
                i += 1
        updated.append(out)
    return updated, replacements


def apply_pair_until_stable(corpus, pair, new_symbol, max_passes=1000):
    current = corpus
    passes = 0
    total_replacements = 0
    for _ in range(max_passes):
        current, replacements = replace_pair_once(current, pair, new_symbol)
        passes += 1
        total_replacements += replacements
        if replacements == 0:
            break
    return current, passes, total_replacements


def segment_to_pair_symbols(sentence, pair_to_symbol):
    segmented = []
    i = 0
    while i < len(sentence):
        if i + 1 < len(sentence) and (sentence[i], sentence[i + 1]) in pair_to_symbol:
            segmented.append(pair_to_symbol[(sentence[i], sentence[i + 1])])
            i += 2
        else:
            return None, i
    return segmented, None


def corpus_stats(corpus):
    lengths = np.array([len(sent) for sent in corpus], dtype=np.int32)
    return {
        'num_sentences': len(corpus),
        'min_len': int(lengths.min()),
        'max_len': int(lengths.max()),
        'mean_len': float(lengths.mean()),
        'median_len': float(np.median(lengths)),
    }


def write_symbol_corpus(corpus, path):
    with open(path, 'w', encoding='utf-8') as f:
        for sent in corpus:
            f.write(' '.join(sent) + '\n')


print('Utilities loaded.')
print(f'  Base preterminals: {BASE_PRETERMS}')


In [ ]:
CORPUS_PATH = 'sample/pcfg3_10k.txt'
PAIR_CORPUS_PATH = 'sample/pcfg3_10k_lexicalcollapsed.txt'
FAMILY_CORPUS_PATH = 'sample/pcfg3_10k_familycollapsed.txt'

LEXICAL_PAIR_SYMBOLS = collections.OrderedDict([
    (('A', 'B'), 'LP_AB'),
    (('A', 'M'), 'LP_AM'),
    (('C', 'D'), 'LP_CD'),
    (('E', 'F'), 'LP_EF'),
    (('E', 'I'), 'LP_EI'),
    (('G', 'H'), 'LP_GH'),
    (('I', 'J'), 'LP_IJ'),
    (('K', 'L'), 'LP_KL'),
    (('M', 'N'), 'LP_MN'),
    (('O', 'P'), 'LP_OP'),
    (('O', 'U'), 'LP_OU'),
    (('PT_S', 'T'), 'LP_ST'),
    (('Q', 'R'), 'LP_QR'),
    (('U', 'V'), 'LP_UV'),
    (('W', 'X'), 'LP_WX'),
    (('Y', 'Z'), 'LP_YZ'),
])
PAIR_SYMBOL_TO_RHS = {symbol: pair for pair, symbol in LEXICAL_PAIR_SYMBOLS.items()}


FAMILY_GROUPS = collections.OrderedDict([
    ('F_ABCD', ['LP_AB', 'LP_CD']),
    ('F_EFGH', ['LP_EF', 'LP_GH']),
    ('F_EIOU', ['LP_EI', 'LP_OU']),
    ('F_IJKL', ['LP_IJ', 'LP_KL']),
    ('F_MNOP', ['LP_MN', 'LP_OP']),
    ('F_QRST', ['LP_QR', 'LP_ST']),
    ('F_UVWX', ['LP_UV', 'LP_WX']),
    ('F_AM', ['LP_AM']),
    ('F_YZ', ['LP_YZ']),
])

PAIR_TO_FAMILY = {}
for family_symbol, pair_symbols in FAMILY_GROUPS.items():
    for pair_symbol in pair_symbols:
        if pair_symbol in PAIR_TO_FAMILY:
            raise ValueError(f'duplicate family assignment for {pair_symbol}')
        PAIR_TO_FAMILY[pair_symbol] = family_symbol

missing_pairs = [symbol for symbol in PAIR_SYMBOL_TO_RHS if symbol not in PAIR_TO_FAMILY]
if missing_pairs:
    raise ValueError(f'unassigned lexical pairs: {missing_pairs}')

raw_corpus = load_symbol_corpus(CORPUS_PATH)
raw_pair_counts = count_adjacent_pairs(raw_corpus)
raw_stats = corpus_stats(raw_corpus)

ordered_pairs = sorted(
    LEXICAL_PAIR_SYMBOLS.items(),
    key=lambda item: (-raw_pair_counts[item[0]], item[1]),
)

print('Pair inventory and raw frequencies:')
for pair, pair_symbol in ordered_pairs:
    print(f"  {pair_symbol:>6s} -> {pair[0]:>5s} {pair[1]:<5s}  raw_count={raw_pair_counts[pair]:5d}")

print('\nHint-driven family groups:')
for family_symbol, pair_symbols in FAMILY_GROUPS.items():
    print(f"  {family_symbol:<8s} <- {', '.join(pair_symbols)}")

pair_corpus = []
family_corpus = []
failed_segmentations = []
pair_symbol_counts = collections.Counter()
family_symbol_counts = collections.Counter()
family_pair_choice_counts = {
    family_symbol: collections.Counter()
    for family_symbol in FAMILY_GROUPS
}

for sent_idx, sent in enumerate(raw_corpus):
    segmented_pairs, fail_pos = segment_to_pair_symbols(sent, LEXICAL_PAIR_SYMBOLS)
    if segmented_pairs is None:
        failed_segmentations.append({
            'sentence_index': sent_idx,
            'fail_pos': fail_pos,
            'sentence': sent,
        })
        continue

    pair_corpus.append(segmented_pairs)
    pair_symbol_counts.update(segmented_pairs)

    family_sent = [PAIR_TO_FAMILY[pair_symbol] for pair_symbol in segmented_pairs]
    family_corpus.append(family_sent)
    family_symbol_counts.update(family_sent)

    for pair_symbol, family_symbol in zip(segmented_pairs, family_sent):
        family_pair_choice_counts[family_symbol][pair_symbol] += 1

pair_stats = corpus_stats(pair_corpus)
family_stats = corpus_stats(family_corpus)
covered_sentences = len(pair_corpus)
dropped_sentences = len(failed_segmentations)
coverage_pct = 100.0 * covered_sentences / len(raw_corpus)

write_symbol_corpus(pair_corpus, PAIR_CORPUS_PATH)
write_symbol_corpus(family_corpus, FAMILY_CORPUS_PATH)

FAMILY_TO_PAIR_PROBS = collections.OrderedDict()
for family_symbol, pair_symbols in FAMILY_GROUPS.items():
    counts = family_pair_choice_counts[family_symbol]
    total = sum(counts.values())
    if total == 0:
        continue
    FAMILY_TO_PAIR_PROBS[family_symbol] = collections.OrderedDict(
        (pair_symbol, counts[pair_symbol] / total)
        for pair_symbol in pair_symbols
        if counts[pair_symbol] > 0
    )

print('\nRaw corpus stats:', raw_stats)
print('Pair-only corpus stats:', pair_stats)
print('Family-only corpus stats:', family_stats)
print(f'Covered sentences: {covered_sentences} / {len(raw_corpus)} ({coverage_pct:.2f}%)')
print(f'Dropped sentences: {dropped_sentences}')
print(f'Pair-only corpus written to   {PAIR_CORPUS_PATH}')
print(f'Family-only corpus written to {FAMILY_CORPUS_PATH}')

if failed_segmentations:
    print('\nFirst 5 uncovered sentences:')
    for item in failed_segmentations[:5]:
        fail_pos = item['fail_pos']
        sent = item['sentence']
        context = ' '.join(sent[max(0, fail_pos - 2):fail_pos + 3])
        print(
            f"  idx={item['sentence_index']:5d}  fail_pos={fail_pos:2d}  "
            f"token={sent[fail_pos]!r}  context={context}"
        )


In [ ]:
present_pair_symbols = [symbol for symbol in LEXICAL_PAIR_SYMBOLS.values() if pair_symbol_counts[symbol] > 0]
present_families = [family_symbol for family_symbol in FAMILY_GROUPS if family_symbol in family_symbol_counts]
family_adjacent_counts = count_adjacent_pairs(family_corpus)

print(f'Observed pair symbols after full segmentation ({len(present_pair_symbols)}):')
print(present_pair_symbols)

print(f'\nObserved family symbols after hint-driven grouping ({len(present_families)}):')
print(present_families)

print('\nLength profile:')
print(f"  raw mean/median/max:    {raw_stats['mean_len']:.2f} / {raw_stats['median_len']:.0f} / {raw_stats['max_len']}")
print(f"  pair mean/median/max:   {pair_stats['mean_len']:.2f} / {pair_stats['median_len']:.0f} / {pair_stats['max_len']}")
print(f"  family mean/median/max: {family_stats['mean_len']:.2f} / {family_stats['median_len']:.0f} / {family_stats['max_len']}")

raw_obs = len(BASE_PRETERMS)
pair_obs = len(present_pair_symbols)
family_obs = len(present_families)
print('\nDense observed child-pair search space:')
print(f'  raw observed inventory:    {raw_obs} symbols -> {raw_obs * raw_obs:,} child pairs')
print(f'  lexical-pair inventory:    {pair_obs} symbols -> {pair_obs * pair_obs:,} child pairs')
print(f'  family-symbol inventory:   {family_obs} symbols -> {family_obs * family_obs:,} child pairs')

print('\nFamily-to-base-pair mixtures:')
for family_symbol in present_families:
    rhs_probs = FAMILY_TO_PAIR_PROBS[family_symbol]
    rhs_str = ', '.join(f'{pair_symbol} ({prob:.3f})' for pair_symbol, prob in rhs_probs.items())
    print(f'  {family_symbol:<8s} -> {rhs_str}')

print('\nTop 30 adjacent family pairs after grouping:')
for (left, right), count in family_adjacent_counts.most_common(30):
    print(f'  {left:<8s} {right:<8s}  count={count}')

print('\nFirst pair-only sentence:')
print(' '.join(pair_corpus[0]))
print('\nFirst family-only sentence:')
print(' '.join(family_corpus[0]))


In [ ]:
class SimplePCFG_EM:
    def __init__(self, observed_symbols, n_nt):
        self.observed_symbols = list(observed_symbols)
        self.obs_to_idx = {sym: i for i, sym in enumerate(self.observed_symbols)}
        self.n_obs = len(self.observed_symbols)
        self.n_nt = n_nt
        self.N = self.n_obs + n_nt
        self.S = self.n_obs
        self.rules = np.zeros((n_nt, self.N, self.N), dtype=np.float64)

    def sym_name(self, idx):
        if idx < self.n_obs:
            return self.observed_symbols[idx]
        if idx == self.S:
            return 'S'
        return f"NT{idx - self.n_obs}"

    def encode_corpus(self, symbol_corpus):
        encoded = []
        for sent in symbol_corpus:
            encoded.append(np.array([self.obs_to_idx[sym] for sym in sent], dtype=np.int32))
        return encoded

    def init_random(self, seed=42):
        rng = np.random.default_rng(seed)
        self.rules[:] = rng.random(self.rules.shape)
        self._normalise()

    def clone(self):
        copied = SimplePCFG_EM(self.observed_symbols, self.n_nt)
        copied.rules = self.rules.copy()
        return copied

    def _normalise(self):
        for a in range(self.n_nt):
            total = self.rules[a].sum()
            if total > 0:
                self.rules[a] /= total

    def inside(self, sent):
        n = len(sent)
        alpha = np.zeros((self.N, n, n), dtype=np.float64)
        for i in range(n):
            alpha[sent[i], i, i] = 1.0

        for width in range(2, n + 1):
            for i in range(n - width + 1):
                j = i + width - 1
                for k in range(i, j):
                    left = alpha[:, i, k]
                    right = alpha[:, k + 1, j]
                    outer = np.outer(left, right)
                    alpha[self.n_obs:, i, j] += np.einsum('abc,bc->a', self.rules, outer)
        return alpha

    def outside(self, sent, alpha):
        n = len(sent)
        beta = np.zeros((self.N, n, n), dtype=np.float64)
        beta[self.S, 0, n - 1] = 1.0

        for width in range(n, 1, -1):
            for i in range(n - width + 1):
                j = i + width - 1
                parent_beta = beta[self.n_obs:, i, j]
                if not np.any(parent_beta):
                    continue
                for k in range(i, j):
                    left_alpha = alpha[:, i, k]
                    right_alpha = alpha[:, k + 1, j]

                    right_weighted = np.einsum('alr,r->al', self.rules, right_alpha)
                    beta[:, i, k] += parent_beta @ right_weighted

                    left_weighted = np.einsum('alr,l->ar', self.rules, left_alpha)
                    beta[:, k + 1, j] += parent_beta @ left_weighted
        return beta

    def em_step(self, sentences, max_len=30):
        counts = np.zeros_like(self.rules)
        ll = 0.0
        used = 0

        for sent in sentences:
            n = len(sent)
            if n < 2 or n > max_len:
                continue

            alpha = self.inside(sent)
            sentence_prob = alpha[self.S, 0, n - 1]
            if sentence_prob < 1e-300:
                continue

            ll += np.log(sentence_prob)
            used += 1
            beta = self.outside(sent, alpha)

            for width in range(2, n + 1):
                for i in range(n - width + 1):
                    j = i + width - 1
                    parent_beta = beta[self.n_obs:, i, j]
                    if not np.any(parent_beta):
                        continue
                    for k in range(i, j):
                        left = alpha[:, i, k]
                        right = alpha[:, k + 1, j]
                        outer = np.outer(left, right)
                        counts += (
                            parent_beta[:, None, None] * self.rules * outer[None, :, :]
                        ) / sentence_prob

        for a in range(self.n_nt):
            total = counts[a].sum()
            if total > 0:
                self.rules[a] = counts[a] / total

        return ll, used

    def prune(self, threshold=0.02):
        self.rules[self.rules < threshold] = 0.0
        self._normalise()

    def count_rules(self, threshold=1e-10):
        return int(np.sum(self.rules > threshold))


def export_expanded_csv(model, output_path, family_to_pair_probs, pair_symbol_to_rhs, threshold=1e-10):
    rows = []
    rule_id = 1

    for base_symbol in BASE_PRETERMS:
        rows.append({
            'ID': rule_id,
            'LHS': base_symbol,
            'LHS Type': 'preterminal',
            'RHS': SYMBOL_TO_WORD[base_symbol],
            'Probability': 1.0,
        })
        rule_id += 1

    for family_symbol, rhs_probs in family_to_pair_probs.items():
        if family_symbol not in model.obs_to_idx:
            continue
        for pair_symbol, prob in rhs_probs.items():
            left, right = pair_symbol_to_rhs[pair_symbol]
            rows.append({
                'ID': rule_id,
                'LHS': family_symbol,
                'LHS Type': 'nonterminal',
                'RHS': f'{left} {right}',
                'Probability': round(float(prob), 6),
            })
            rule_id += 1

    for a in range(model.n_nt):
        for b in range(model.N):
            for c in range(model.N):
                prob = model.rules[a, b, c]
                if prob > threshold:
                    rows.append({
                        'ID': rule_id,
                        'LHS': model.sym_name(a + model.n_obs),
                        'LHS Type': 'nonterminal',
                        'RHS': f"{model.sym_name(b)} {model.sym_name(c)}",
                        'Probability': round(float(prob), 6),
                    })
                    rule_id += 1

    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['ID', 'LHS', 'LHS Type', 'RHS', 'Probability'])
        writer.writeheader()
        writer.writerows(rows)

    return rows


def export_seed_pcfg(model, output_path, family_to_pair_probs, pair_symbol_to_rhs, threshold=1e-10):
    lines = []
    for family_symbol, rhs_probs in family_to_pair_probs.items():
        if family_symbol not in model.obs_to_idx:
            continue
        for pair_symbol, prob in rhs_probs.items():
            left, right = pair_symbol_to_rhs[pair_symbol]
            lines.append(f'{family_symbol} -> {left} {right} [{prob:.6f}]')
    for a in range(model.n_nt):
        lhs = model.sym_name(a + model.n_obs)
        for b in range(model.N):
            for c in range(model.N):
                prob = model.rules[a, b, c]
                if prob > threshold:
                    rhs = f'{model.sym_name(b)} {model.sym_name(c)}'
                    lines.append(f'{lhs} -> {rhs} [{prob:.6f}]')
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines) + '\n')


print('Basic EM utilities loaded.')


In [ ]:
observed_symbols = [family_symbol for family_symbol in FAMILY_GROUPS if family_symbol in present_families]

N_NT = 12
SEED = 42
EM_ITERS = 18
EM_BATCH = 2000  # set to 0 for full-batch EM
MAX_LEN = 24

model = SimplePCFG_EM(observed_symbols, N_NT)
model.init_random(seed=SEED)
encoded_corpus = model.encode_corpus(family_corpus)
eligible = [sent for sent in encoded_corpus if 2 <= len(sent) <= MAX_LEN]

if not eligible:
    raise ValueError(f'No eligible sentences found for MAX_LEN={MAX_LEN}')

print(f'Observed family inventory for EM: {len(observed_symbols)}')
print(observed_symbols)
print(f'Rule tensor size: {model.n_nt} x {model.N} x {model.N} = {model.n_nt * model.N * model.N:,}')
print(f'EM config: {EM_ITERS} iters, batch={EM_BATCH or "all"}, max_len={MAX_LEN}')
print(f'Eligible sentences: {len(eligible)}')

history = []
rng = np.random.default_rng(SEED)

for it in range(1, EM_ITERS + 1):
    if EM_BATCH and EM_BATCH < len(eligible):
        idxs = rng.choice(len(eligible), size=EM_BATCH, replace=False)
        batch_sents = [eligible[i] for i in idxs]
    else:
        batch_sents = eligible

    t0 = time.time()
    ll, used = model.em_step(batch_sents, max_len=MAX_LEN)
    dt = time.time() - t0
    avg_ll = ll / max(used, 1)
    nr = model.count_rules()
    history.append({
        'iter': it,
        'll': ll,
        'avg_ll': avg_ll,
        'used': used,
        'rules': nr,
        'time': dt,
    })
    print(
        f"  [{it:3d}/{EM_ITERS}]  avg_LL={avg_ll:8.3f}  "
        f"sents={used:4d}  rules={nr:5d}  {dt:5.1f}s"
    )

print(f"\nTraining complete. Active rules: {model.count_rules()}")


In [ ]:
OUTPUT_CSV = 'pcfg3.csv'
OUTPUT_PCFG = 'task3_seed.pcfg'
FINAL_PRUNE_THRESHOLD = 0.02
SHOW_TOP_N = 60

export_model = model.clone()
before = export_model.count_rules()
export_model.prune(threshold=FINAL_PRUNE_THRESHOLD)
after = export_model.count_rules()

print(f'Final prune: {before} -> {after} rules at threshold {FINAL_PRUNE_THRESHOLD:.3f}')
rows = export_expanded_csv(export_model, OUTPUT_CSV, FAMILY_TO_PAIR_PROBS, PAIR_SYMBOL_TO_RHS)
export_seed_pcfg(export_model, OUTPUT_PCFG, FAMILY_TO_PAIR_PROBS, PAIR_SYMBOL_TO_RHS)

family_rows = [
    row for row in rows
    if row['LHS Type'] == 'nonterminal' and row['LHS'] in observed_symbols
]
family_rows = sorted(family_rows, key=lambda row: (row['LHS'], -row['Probability'], row['RHS']))

latent_rows = [
    row for row in rows
    if row['LHS Type'] == 'nonterminal' and row['LHS'] not in observed_symbols
]
latent_rows = sorted(latent_rows, key=lambda row: row['Probability'], reverse=True)

print(f'Expanded grammar written to {OUTPUT_CSV}')
print(f'Seed grammar string written to {OUTPUT_PCFG}')
print(f"Total rows: {len(rows)}  Nonterminal rows: {len(family_rows) + len(latent_rows)}")

print('\nFamily expansions:')
for row in family_rows:
    print(f"  {row['LHS']:>8s} -> {row['RHS']:<18s}  P={row['Probability']:.4f}")

print(f'\nTop {min(SHOW_TOP_N, len(latent_rows))} latent rules:')
for row in latent_rows[:SHOW_TOP_N]:
    print(f"  {row['LHS']:>8s} -> {row['RHS']:<18s}  P={row['Probability']:.4f}")


In [ ]:
# Evaluate exact pair-resolved strings under the exported family grammar.
# The higher-level inside score is computed on family symbols, then we add the
# log-probability of each family choosing the observed base pair.

EVAL_SENTENCES = min(10000, len(family_corpus))  # set to len(family_corpus) for a full pass
eval_family_corpus = family_corpus[:EVAL_SENTENCES]
eval_pair_corpus = pair_corpus[:EVAL_SENTENCES]
encoded_eval = export_model.encode_corpus(eval_family_corpus)

parsed = 0
failed = 0
total_log2_prob = 0.0
fail_by_len = collections.Counter()

for idx, (family_sent, pair_sent, encoded_sent) in enumerate(zip(eval_family_corpus, eval_pair_corpus, encoded_eval), start=1):
    alpha = export_model.inside(encoded_sent)
    family_prob = alpha[export_model.S, 0, len(encoded_sent) - 1]

    if family_prob > 1e-300:
        lexical_log2 = 0.0
        lexical_ok = True
        for family_symbol, pair_symbol in zip(family_sent, pair_sent):
            pair_prob = FAMILY_TO_PAIR_PROBS[family_symbol].get(pair_symbol, 0.0)
            if pair_prob <= 1e-300:
                lexical_ok = False
                break
            lexical_log2 += np.log2(pair_prob)

        if lexical_ok:
            total_log2_prob += np.log2(family_prob) + lexical_log2
            parsed += 1
        else:
            failed += 1
            fail_by_len[len(pair_sent)] += 1
    else:
        failed += 1
        fail_by_len[len(pair_sent)] += 1

    if idx % 200 == 0:
        print(f'Processed {idx}/{EVAL_SENTENCES} family-grouped sentences...')

coverage = 100.0 * parsed / max(parsed + failed, 1)
avg_log2_prob = total_log2_prob / max(parsed, 1)

print('\n' + '=' * 40)
print('FAMILY-LEVEL EVALUATION')
print('=' * 40)
print(f'Pair-segmented sentences available:  {len(pair_corpus)}')
print(f'Dropped before segmentation:        {dropped_sentences}')
print(f'Evaluated pair-resolved strings:    {EVAL_SENTENCES}')
print(f'Successfully scored:                {parsed} / {EVAL_SENTENCES}')
print(f'Failed:                             {failed}')
print(f'Coverage:                           {coverage:.2f}%')
print(f'Average Log2 Likelihood:            {avg_log2_prob:.4f}')

if failed > 0:
    print('\nFailures by pair length:')
    for length, count in fail_by_len.most_common(10):
        print(f'  len={length:2d}: {count} failures')
